# 06 — Extrapolation des clusters CAH à la base Karafun

## Objectif et périmètre

Le projet vise à attribuer les cinq groupes de référence aux **341 092 individus** de la base brute, sans réentraîner l’ACM ni recalculer la CAH.

Cette première partie traite **les 68 218 individus de l’échantillon ACM** : conservation des **20 000 étiquettes CAH** et prédiction des **48 218 autres individus** par la régression logistique multinomiale déjà ajustée. **La projection et la classification des 272 874 individus hors échantillon ACM restent à réaliser** et ne sont pas annoncées comme effectuées par ce notebook.

**Prérequis :** les exports des notebooks 02, 03 et 05, inchangés et issus de la même base source et du même ordre de lignes.

## 1. Configuration et chargement des artefacts

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

RACINE = Path.cwd().resolve()
if RACINE.name == "notebooks":
    RACINE = RACINE.parent
if not (RACINE / "src").is_dir():
    raise FileNotFoundError("Exécuter le notebook depuis la racine du dépôt ou le dossier notebooks/.")

import sys
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src.classification_hierarchique import charger_coordonnees_et_identifiants
from src.classification_supervisee import NOMS_METIER_K5, charger_reference_cah

TAILLE_ACM_ATTENDUE = 68218
TAILLE_CAH_ATTENDUE = 20000
TAILLE_HORS_CAH_ATTENDUE = 48218
NOMBRE_AXES = 4
NOMBRE_GROUPES = 5
DOSSIER_SORTIE = RACINE / "resultats" / "extrapolation"

coordonnees, identifiants, metadonnees_acm = charger_coordonnees_et_identifiants(RACINE)
X_reference, y_reference, partitions, colonnes_axes, parametres_cah, _ = charger_reference_cah(
    RACINE, taille=TAILLE_CAH_ATTENDUE, axes=NOMBRE_AXES, graine=42, k=NOMBRE_GROUPES
)
chemin_modele = RACINE / "modeles" / "classifieur_clusters_k5.joblib"
chemin_metadonnees = RACINE / "modeles" / "classifieur_clusters_k5_metadonnees.json"
with chemin_metadonnees.open(encoding="utf-8") as fichier:
    metadonnees_classifieur = json.load(fichier)
modele = joblib.load(chemin_modele)

assert len(coordonnees) == len(identifiants) == TAILLE_ACM_ATTENDUE
assert len(partitions) == TAILLE_CAH_ATTENDUE
assert metadonnees_acm["iteration"] == 3
assert metadonnees_classifieur["modele_selectionne"] == "regression_logistique_multinomiale"
assert metadonnees_classifieur["nombre_axes"] == 4
assert metadonnees_classifieur["nombre_groupes"] == 5
assert metadonnees_classifieur["colonnes_axes"] == colonnes_axes
assert len(metadonnees_acm["variables_actives"]) == 10
assert len(metadonnees_acm["modalites_actives"]) == 42
assert np.array_equal(np.asarray(modele.classes_), np.arange(1, 6))
assert list(coordonnees.columns[:4]) == colonnes_axes
print(f"Échantillon ACM : {len(coordonnees):,} individus")
print(f"Partition CAH : {len(partitions):,} individus")
print(f"Classifieur retenu : {metadonnees_classifieur['modele_selectionne']}")

Échantillon ACM : 68,218 individus
Partition CAH : 20,000 individus
Classifieur retenu : regression_logistique_multinomiale


## 2. Correspondance et séparation sans ambiguïté

Les positions CAH correspondent aux **positions dans l’échantillon ACM**, non aux indices bruts. Les identifiants sont conservés et aucune jointure approximative n’est autorisée.

In [2]:
positions_cah = partitions["position_echantillon"].to_numpy(dtype=int)
assert len(np.unique(positions_cah)) == TAILLE_CAH_ATTENDUE
assert np.array_equal(
    identifiants.iloc[positions_cah]["position_echantillon"].to_numpy(), positions_cah
)

masque_cah = np.zeros(len(coordonnees), dtype=bool)
masque_cah[positions_cah] = True
positions_a_predire = np.flatnonzero(~masque_cah)
assert len(positions_a_predire) == TAILLE_HORS_CAH_ATTENDUE
assert len(np.intersect1d(positions_cah, positions_a_predire)) == 0
assert len(np.union1d(positions_cah, positions_a_predire)) == len(coordonnees)

colonnes_identification = list(identifiants.columns)
reference = identifiants.iloc[positions_cah].copy().reset_index(drop=True)
reference["groupe_k5"] = y_reference.to_numpy(dtype=int)
reference["origine_affectation"] = "CAH_reference"
reference["probabilite_max"] = np.nan
for groupe in range(1, 6):
    reference[f"probabilite_G{groupe}"] = np.nan

X_a_predire = coordonnees.iloc[positions_a_predire][colonnes_axes].copy()
assert X_a_predire.shape == (TAILLE_HORS_CAH_ATTENDUE, 4)
assert np.isfinite(X_a_predire.to_numpy(dtype=float)).all()
print("Correspondance validée ; lignes à prédire :", len(X_a_predire))

Correspondance validée ; lignes à prédire : 48218


## 3. Prédiction par la régression logistique multinomiale sauvegardée

Ici, seul `predict()` / `predict_proba()` est appelé : **aucun `fit()` n’est effectué**.

In [3]:
etiquettes_predites = np.asarray(modele.predict(X_a_predire), dtype=int)
probabilites = np.asarray(modele.predict_proba(X_a_predire), dtype=float)
assert probabilites.shape == (TAILLE_HORS_CAH_ATTENDUE, 5)
assert np.isfinite(probabilites).all()
assert np.allclose(probabilites.sum(axis=1), 1.0, atol=1e-8)
assert set(np.unique(etiquettes_predites)).issubset({1, 2, 3, 4, 5})
assert np.array_equal(etiquettes_predites, np.asarray(modele.classes_)[probabilites.argmax(axis=1)])

predictions = identifiants.iloc[positions_a_predire].copy().reset_index(drop=True)
predictions["groupe_k5"] = etiquettes_predites
predictions["origine_affectation"] = "regression_logistique_multinomiale"
predictions["probabilite_max"] = probabilites.max(axis=1)
for indice, groupe in enumerate(modele.classes_):
    predictions[f"probabilite_G{int(groupe)}"] = probabilites[:, indice]

reference["nom_segment"] = reference["groupe_k5"].map(NOMS_METIER_K5)
predictions["nom_segment"] = predictions["groupe_k5"].map(NOMS_METIER_K5)
assert reference["nom_segment"].notna().all()
assert predictions["nom_segment"].notna().all()
display(predictions.head())

,position_echantillon,id_anonyme,groupe_k5,origine_affectation,probabilite_max,probabilite_G1,probabilite_G2,probabilite_G3,probabilite_G4,probabilite_G5,nom_segment
0,1,73f52ec545df5ce3b1a3fd90abdd76ec5bd4137fe14bdd...,1,regression_logistique_multinomiale,0.994069,0.994069,0.005923,1.265931e-12,1.950195e-06,0.000006,Clients fidèles à forte valeur
1,2,9e40eee05b42e2d8c99ff8a972ff66f47db2614f7fc966...,4,regression_logistique_multinomiale,0.999581,0.000005,0.000402,2.228377e-08,9.995807e-01,0.000013,Clients intermédiaires souvent réactivés
2,3,0b8146f0a00c5cf4c3962373218cb5309a8f64b09ac645...,5,regression_logistique_multinomiale,0.999997,0.000002,0.000001,1.786441e-16,5.300161e-13,0.999997,Clients récents en phase d'intégration
3,4,0c049beaeada4fc987a933703e1a87e4d290fbf81d9343...,4,regression_logistique_multinomiale,0.997393,0.000007,0.000005,1.335325e-10,9.973935e-01,0.002595,Clients intermédiaires souvent réactivés
4,5,8b3c4e2bf6c0214e23f20e50026b56fb7418d69beb62f5...,2,regression_logistique_multinomiale,0.993660,0.004786,0.993660,1.850100e-07,1.257371e-04,0.001428,Clients en churn à faible fidélité


## 4. Fusion des 68 218 affectations et contrôles

In [4]:
colonnes_finales = (
    colonnes_identification + ["groupe_k5", "nom_segment", "origine_affectation", "probabilite_max"]
    + [f"probabilite_G{groupe}" for groupe in range(1, 6)]
)
base_acm_segmentee = pd.concat(
    [reference[colonnes_finales], predictions[colonnes_finales]],
    ignore_index=True,
).sort_values("position_echantillon", kind="stable").reset_index(drop=True)
assert len(base_acm_segmentee) == TAILLE_ACM_ATTENDUE
assert base_acm_segmentee["position_echantillon"].is_unique
assert np.array_equal(base_acm_segmentee["position_echantillon"], np.arange(TAILLE_ACM_ATTENDUE))
assert base_acm_segmentee["groupe_k5"].notna().all()
assert (base_acm_segmentee["origine_affectation"] == "CAH_reference").sum() == TAILLE_CAH_ATTENDUE
assert (base_acm_segmentee["origine_affectation"] == "regression_logistique_multinomiale").sum() == TAILLE_HORS_CAH_ATTENDUE
assert np.array_equal(base_acm_segmentee.iloc[positions_cah]["groupe_k5"].to_numpy(), y_reference.to_numpy())

bilan = (base_acm_segmentee.groupby(["origine_affectation", "groupe_k5", "nom_segment"])
         .size().rename("effectif").reset_index())
bilan["part_dans_origine_pct"] = 100 * bilan["effectif"] / bilan.groupby("origine_affectation")["effectif"].transform("sum")
display(bilan)
print("Clients segmentés dans l'échantillon ACM :", len(base_acm_segmentee))
print("Probabilité maximale médiane (prédictions) :", round(float(predictions["probabilite_max"].median()), 4))
print("Part des prédictions avec probabilité maximale < 0,60 :", round(float((predictions["probabilite_max"] < 0.60).mean() * 100), 2), "%")

,origine_affectation,groupe_k5,nom_segment,effectif,part_dans_origine_pct
0,CAH_reference,1,Clients fidèles à forte valeur,5134,25.670000
1,CAH_reference,2,Clients en churn à faible fidélité,4152,20.760000
2,CAH_reference,3,Clients sans activité observée,2201,11.005000
3,CAH_reference,4,Clients intermédiaires souvent réactivés,4060,20.300000
4,CAH_reference,5,Clients récents en phase d'intégration,4453,22.265000
5,regression_logistique_multinomiale,1,Clients fidèles à forte valeur,12735,26.411299
6,regression_logistique_multinomiale,2,Clients en churn à faible fidélité,10010,20.759882
7,regression_logistique_multinomiale,3,Clients sans activité observée,5374,11.145215
8,regression_logistique_multinomiale,4,Clients intermédiaires souvent réactivés,9546,19.797586
9,regression_logistique_multinomiale,5,Clients récents en phase d'intégration,10553,21.886018


Clients segmentés dans l'échantillon ACM : 68218
Probabilité maximale médiane (prédictions) : 0.9994
Part des prédictions avec probabilité maximale < 0,60 : 3.52 %


## 5. Export des résultats intermédiaires

In [5]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
predictions.to_csv(DOSSIER_SORTIE / "clients_acm_hors_cah_predits.csv", index=False)
base_acm_segmentee.to_csv(DOSSIER_SORTIE / "echantillon_acm_segmente.csv", index=False)
bilan.to_csv(DOSSIER_SORTIE / "bilan_echantillon_acm.csv", index=False)
print("Exports intermédiaires enregistrés dans :", DOSSIER_SORTIE)

Exports intermédiaires enregistrés dans : C:\Users\SebastianAraya\Documents\Documentos\LILLE\Master\3S\Clustering\clustering\resultats\extrapolation


## 6. Conclusion provisoire et suite de l'étape 3

Cette première partie préserve les étiquettes des 20 000 clients de référence CAH et affecte les 48 218 autres individus de l'échantillon ACM avec le classifieur multinomial sauvegardé.

**Ce résultat ne couvre pas encore l'intégralité de la base originale.** La prochaine partie devra :

1. Charger les 341 092 lignes brutes et identifier les 272 874 individus absents de l'échantillon ACM, avec une correspondance vérifiable.
2. Réappliquer à ces individus la préparation et les modalités définitives de l'ACM 3.
3. Les projeter sur les axes sauvegardés sans réajuster l'ACM ; contrôler que la projection reproduit les coordonnées déjà exportées sur des individus connus.
4. Prédire leurs groupes et fusionner les trois provenances (**CAH**, **ACM hors CAH**, **hors ACM**), sans doublon ni perte de ligne.
5. Exporter et vérifier la table finale de **341 092 individus**, en conservant l'origine de chaque affectation.

**Attention :** `indice_ligne_source` est une clé technique dépendant de l'ordre du CSV original si `id_anonyme` n'existe pas. Une modification de cet ordre invalide les correspondances et impose de reconstituer une clé stable avant la fusion.

Les probabilités affichées mesurent la confiance du classifieur dans la reproduction des groupes de la CAH et non un risque de churn.